# 01 - Exploratory Data Analysis (UCI CKD)

Goals: understand the data, document quality issues, and identify features that leak the label.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw, clean, load_clean, missingness_report
from src.features import NUMERIC, CATEGORICAL, TARGET, LEAKAGE_BROAD

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Raw data and its quality problems

In [ ]:
raw = load_raw()
print(raw.shape)
raw.head()

In [ ]:
# Show the messy values that cleaning has to fix
for col in raw.select_dtypes(include=["object", "string"]).columns:
    vals = raw[col].dropna().unique()
    print(col, "->", list(vals)[:8])

## 2. Cleaned data

In [ ]:
df = load_clean(refresh=True)
print(df.shape)
df.describe().T

## 3. Class balance

In [ ]:
counts = df[TARGET].value_counts().rename({1: "CKD", 0: "Not CKD"})
print(counts)
counts.plot(kind="bar", rot=0, title="Class balance")
plt.show()

## 4. Missingness

Decide later whether missingness is informative (e.g. are some tests only ordered for sicker patients?).

In [ ]:
rep = missingness_report(df)
display(rep)
rep["pct"].plot(kind="barh", figsize=(6, 7), title="% missing by column")
plt.show()

In [ ]:
# Is missingness related to the label? (missing-at-random check)
miss_flags = df.drop(columns=[TARGET]).isna().astype(int)
by_class = miss_flags.groupby(df[TARGET]).mean().T.rename(columns={0: "notckd", 1: "ckd"})
by_class["gap"] = (by_class["ckd"] - by_class["notckd"]).abs()
by_class.sort_values("gap", ascending=False).head(10)

## 5. Distributions by class

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for ax, col in zip(axes.ravel(), NUMERIC):
    sns.kdeplot(data=df, x=col, hue=TARGET, common_norm=False, ax=ax, warn_singular=False)
    ax.set_title(col)
for ax in axes.ravel()[len(NUMERIC):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Correlations

In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0, annot=False)
plt.title("Correlation matrix")
plt.show()

## 7. Leakage check

How well does a **single** feature separate the classes? Features with AUC near 1.0 on their own (serum creatinine, hemoglobin, specific gravity...) make the full-feature task trivial.

In [ ]:
from sklearn.metrics import roc_auc_score

rows = []
for col in [c for c in df.columns if c != TARGET]:
    sub = df[[col, TARGET]].dropna()
    if sub[col].nunique() < 2:
        continue
    auc = roc_auc_score(sub[TARGET], sub[col])
    rows.append((col, max(auc, 1 - auc), len(sub)))

single = (pd.DataFrame(rows, columns=["feature", "single_feature_auc", "n_used"])
            .sort_values("single_feature_auc", ascending=False))
single.head(12)

In [ ]:
single.set_index("feature")["single_feature_auc"].head(12)[::-1].plot(
    kind="barh", title="Single-feature AUC (higher = more leakage-prone)")
plt.axvline(0.9, color="red", ls="--")
plt.show()

## Notes to fill in

- Which columns have the most missing data, and how will you impute them?
- Which features separate the classes almost perfectly on their own?
- What does this mean for the claim "99% accuracy"?